# Housing Price Prediction Model - Interactive Notebook

This notebook provides an interactive way to explore and train the housing price prediction model.

## 1. Setup and Imports

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("All packages installed!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Generate sample housing dataset
np.random.seed(42)
n_samples = 500

df = pd.DataFrame({
    'square_feet': np.random.uniform(800, 5000, n_samples),
    'bedrooms': np.random.randint(1, 6, n_samples),
    'bathrooms': np.random.uniform(1, 4, n_samples),
    'age_years': np.random.uniform(0, 100, n_samples),
    'garage_spaces': np.random.randint(0, 4, n_samples),
    'lot_size': np.random.uniform(2000, 15000, n_samples),
    'condition': np.random.randint(1, 6, n_samples),
    'location_quality': np.random.randint(1, 10, n_samples),
})

# Generate target variable
df['price'] = (
    df['square_feet'] * 150 +
    df['bedrooms'] * 30000 +
    df['bathrooms'] * 40000 -
    df['age_years'] * 500 +
    df['garage_spaces'] * 20000 +
    df['lot_size'] * 2 +
    df['condition'] * 25000 +
    df['location_quality'] * 40000 +
    np.random.normal(0, 50000, n_samples)
)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

## 3. Exploratory Data Analysis

In [ ]:
# Correlation analysis
correlation_with_price = df.corr()['price'].sort_values(ascending=False)
print("Correlation with Price:")
print(correlation_with_price)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Housing Features')
plt.tight_layout()
plt.show()

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['price'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of House Prices')

axes[1].scatter(df['square_feet'], df['price'], alpha=0.5)
axes[1].set_xlabel('Square Feet')
axes[1].set_ylabel('Price')
axes[1].set_title('Price vs Square Footage')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('price', axis=1)
y = df['price']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessing completed!")

## 5. Model Training

In [ ]:
# Train Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
print("✓ Linear Regression trained")

# Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("✓ Random Forest trained")

# Train Gradient Boosting
print("Training Gradient Boosting...")
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb_model.fit(X_train, y_train)
print("✓ Gradient Boosting trained")

## 6. Model Evaluation

In [ ]:
# Evaluate models
models = {
    'Linear Regression': (lr_model, X_train_scaled, X_test_scaled),
    'Random Forest': (rf_model, X_train, X_test),
    'Gradient Boosting': (gb_model, X_train, X_test)
}

results = []

for model_name, (model, X_tr, X_te) in models.items():
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    print(f"\n{model_name}:")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²:  {test_r2:.4f}")
    print(f"  Test RMSE: ${test_rmse:,.2f}")
    print(f"  Test MAE:  ${test_mae:,.2f}")
    
    results.append({
        'Model': model_name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae
    })

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("Model Comparison Summary")
print("="*70)
print(results_df.to_string(index=False))

## 7. Feature Importance

In [ ]:
# Random Forest Feature Importance
rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

print("Random Forest Feature Importance:")
print(rf_importance.sort_values(ascending=False))

In [ ]:
# Plot feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
rf_importance.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest - Feature Importance')
axes[0].set_xlabel('Importance')

# Gradient Boosting
gb_importance = pd.Series(
    gb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

gb_importance.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Gradient Boosting - Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 8. Predictions Visualization

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, (model_name, (model, X_tr, X_te)) in enumerate(models.items()):
    y_pred = model.predict(X_te)
    r2 = r2_score(y_test, y_pred)
    
    axes[idx].scatter(y_test, y_pred, alpha=0.5)
    axes[idx].plot([y_test.min(), y_test.max()],
                  [y_test.min(), y_test.max()],
                  'r--', lw=2)
    axes[idx].set_xlabel('Actual Price')
    axes[idx].set_ylabel('Predicted Price')
    axes[idx].set_title(f'{model_name}\nR² = {r2:.4f}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Making Predictions on New Data

In [ ]:
# Create a sample house for prediction
sample_house = pd.DataFrame({
    'square_feet': [2500],
    'bedrooms': [4],
    'bathrooms': [2.5],
    'age_years': [10],
    'garage_spaces': [2],
    'lot_size': [8000],
    'condition': [4],
    'location_quality': [8]
})

print("Sample House Features:")
print(sample_house.to_string(index=False))

# Make predictions
sample_scaled = scaler.transform(sample_house)

lr_pred = lr_model.predict(sample_scaled)[0]
rf_pred = rf_model.predict(sample_house)[0]
gb_pred = gb_model.predict(sample_house)[0]

print("\nPredicted Prices:")
print(f"  Linear Regression: ${lr_pred:,.2f}")
print(f"  Random Forest:     ${rf_pred:,.2f}")
print(f"  Gradient Boosting: ${gb_pred:,.2f}")
print(f"\n  Average:          ${(lr_pred + rf_pred + gb_pred) / 3:,.2f}")

## 10. Interactive Prediction

In [ ]:
# Define different house scenarios
scenarios = {
    'Small Apartment': {
        'square_feet': 1000,
        'bedrooms': 1,
        'bathrooms': 1,
        'age_years': 5,
        'garage_spaces': 0,
        'lot_size': 2000,
        'condition': 3,
        'location_quality': 5
    },
    'Average House': {
        'square_feet': 2500,
        'bedrooms': 3,
        'bathrooms': 2,
        'age_years': 20,
        'garage_spaces': 2,
        'lot_size': 8000,
        'condition': 4,
        'location_quality': 7
    },
    'Luxury Home': {
        'square_feet': 5000,
        'bedrooms': 5,
        'bathrooms': 4,
        'age_years': 10,
        'garage_spaces': 3,
        'lot_size': 15000,
        'condition': 5,
        'location_quality': 10
    }
}

predictions_data = []

for scenario_name, features in scenarios.items():
    scenario_df = pd.DataFrame([features])
    scenario_scaled = scaler.transform(scenario_df)
    
    lr_pred = lr_model.predict(scenario_scaled)[0]
    rf_pred = rf_model.predict(scenario_df)[0]
    gb_pred = gb_model.predict(scenario_df)[0]
    avg_pred = (lr_pred + rf_pred + gb_pred) / 3
    
    predictions_data.append({
        'Scenario': scenario_name,
        'Linear Regression': f'${lr_pred:,.0f}',
        'Random Forest': f'${rf_pred:,.0f}',
        'Gradient Boosting': f'${gb_pred:,.0f}',
        'Average': f'${avg_pred:,.0f}'
    })

predictions_summary = pd.DataFrame(predictions_data)
print("\nPrice Predictions for Different Scenarios:")
print(predictions_summary.to_string(index=False))

## 11. Summary and Key Insights

In [ ]:
print("="*70)
print("PROJECT SUMMARY")
print("="*70)

print(f"\n1. Dataset Information:")
print(f"   - Total samples: {len(df)}")
print(f"   - Number of features: {len(X.columns)}")
print(f"   - Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")

print(f"\n2. Best Performing Model:")
best_model_idx = results_df['Test R²'].idxmax()
best_model = results_df.loc[best_model_idx]
print(f"   - Model: {best_model['Model']}")
print(f"   - Test R²: {best_model['Test R²']:.4f}")
print(f"   - Test RMSE: ${best_model['Test RMSE']:,.2f}")

print(f"\n3. Most Important Features:")
top_features = rf_importance.nlargest(3)
for feature, importance in top_features.items():
    print(f"   - {feature}: {importance:.4f}")

print(f"\n4. Average Prediction Error:")
gb_test_pred = gb_model.predict(X_test)
mae = mean_absolute_error(y_test, gb_test_pred)
mape = np.mean(np.abs((y_test - gb_test_pred) / y_test)) * 100
print(f"   - MAE: ${mae:,.2f}")
print(f"   - MAPE: {mape:.2f}%")

print(f"\n5. Model Recommendations:")
print(f"   - Use Gradient Boosting for best accuracy")
print(f"   - Use Random Forest for interpretability")
print(f"   - Use Linear Regression for fast predictions")